# 05 · Aşama 5–6 — Federe öğrenme (FedAvg): IID ve non-IID istemciler
* **İstemciler:** 5 istemci; normal veri (eğitim/doğrulama/kalibrasyon/test) istemcilere bölünür.
  **IID:** rastgele; **non-IID:** `service` kategorisine göre Dirichlet bölmesi (α_D ∈ {1,0; 0,1}, küçük α_D = daha farklı istemciler).
* **Karşılaştırma:** merkezi eğitim, yalnız yerel eğitim, FedAvg — aynı toplam eğitim bütçesiyle.
* **Güven sorusu (S1):** global eşik her istemcide yanlış alarm bütçesini tutuyor mu? İstemci başına kalibrasyon ne değiştiriyor?
* **İletişim maliyeti** = parametre × 4 bayt × 2 yön × istemci × tur.

**Not:** Zaman kısıtı nedeniyle bu aşama tohum 0 ile çalıştırılır (Aşama 1–4 10 tohumla). Sonuçlar keşif amaçlıdır.

In [ ]:
# ---- Kurulum (her defterde aynı): Drive'ı bağla, ortak kodu içe aktar ----
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_DIR = os.environ.get("TRUSTIDS_PROJECT", "/content/drive/MyDrive/TrustFed-TinyIDS")
QUICK = os.environ.get("TRUSTIDS_QUICK", "0") == "1"   # hızlı mod YALNIZCA kodu test etmek içindir
# Devam modu: Colab oturumu kapanırsa, Drive'a kaydedilmiş tohumlar/adımlar atlanır ve kalan yerden devam edilir.
# Her şeyi baştan hesaplamak için: os.environ["TRUSTIDS_RESUME"] = "0"
RESUME = os.environ.get("TRUSTIDS_RESUME", "1") == "1"
sys.path.insert(0, PROJECT_DIR)

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from trustids.config import Config
from trustids import utils

# Veri kümesi seçimi: varsayılan TON_IoT; ikinci veri kümesi için TRUSTIDS_CONFIG=edge_iiotset.json
CONFIG_NAME = os.environ.get("TRUSTIDS_CONFIG", "default.json")
CFG_PATH = Path(PROJECT_DIR) / "configs" / CONFIG_NAME
if CONFIG_NAME != "default.json" and not CFG_PATH.exists():
    raise FileNotFoundError(f"{CFG_PATH} bulunamadı: configs/ klasörünü de Drive'a yükleyin.")
cfg = Config.load(CFG_PATH) if CFG_PATH.exists() else Config(project_dir=PROJECT_DIR)
cfg.project_dir = PROJECT_DIR
if QUICK:
    cfg.train_seeds, cfg.validity_resplits, cfg.epochs = cfg.train_seeds[:2], 30, 15
    cfg.random_control_repeats, cfg.bootstrap_repeats = 10, 200
cfg.check()
ART = cfg.artifacts
FIG = Path(PROJECT_DIR) / cfg.figures_dir; FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Proje klasörü:", PROJECT_DIR, "| hızlı mod:", QUICK, "| devam modu:", RESUME)
print("Drive'da kayıtlı ilerleme:", utils.progress(cfg))

In [ ]:
import tensorflow as tf
from trustids import data as D, models as M, conformal as C, evaluation as E
P = D.load_prepared(ART / "prepared"); X, T = P["X"], P["types"]
FN = P["preprocessor"].feature_names_
spec = cfg.ae_variants[utils.load_json(ART / "ae_selection.json")["chosen"]]
K, ROUNDS, LOCAL_EP, SEED = 5, (3 if QUICK else 40), (1 if QUICK else 5), cfg.train_seeds[0]
FL_DIR = ART / "fl"; FL_DIR.mkdir(exist_ok=True)
RUN = utils.new_run_dir(Path(PROJECT_DIR) / cfg.runs_dir, "final_phase5_fl")
svc = [i for i, f in enumerate(FN) if f.startswith("service=")]
def category(Xs):
    if not svc: return np.array(["tümü"] * len(Xs))
    lab = np.array([FN[svc[i]] for i in Xs[:, svc].argmax(1)], dtype=object)
    lab[Xs[:, svc].max(1) < 0.5] = "service=diğer"; return lab
CAT = {sp: category(X[sp]) for sp in ["benign_train", "benign_val", "benign_cal", "benign_test"]}
print("istemci sayısı", K, "| tur", ROUNDS, "| yerel epoch", LOCAL_EP, "| kategoriler:", pd.Series(CAT["benign_train"]).value_counts().to_dict())

## İstemci bölmeleri (aynı kategori oranları tüm normal bölmelere uygulanır)

In [ ]:
def partition(kind, seed):
    rng = np.random.default_rng(seed)
    cats = sorted(set(np.concatenate(list(CAT.values()))))
    if kind == "IID":
        props = {c: np.full(K, 1 / K) for c in cats}
    else:
        a = float(kind.split("=")[1]); props = {c: rng.dirichlet([a] * K) for c in cats}
    parts = {}
    for sp, lab in CAT.items():
        assign = np.empty(len(lab), dtype=int)
        for c in cats:
            idx = rng.permutation(np.where(lab == c)[0])
            cuts = (np.cumsum(props[c])[:-1] * len(idx)).astype(int)
            for k, chunk in enumerate(np.split(idx, cuts)):
                assign[chunk] = k
        parts[sp] = [np.where(assign == k)[0] for k in range(K)]
    return parts
SCEN = ["IID", "Dirichlet=1.0", "Dirichlet=0.1"]
PARTS = {s: partition(s, SEED) for s in SCEN}
SIZES = pd.DataFrame([{"senaryo": s, "istemci": k, **{sp: len(PARTS[s][sp][k]) for sp in CAT}} for s in SCEN for k in range(K)])
SIZES.to_csv(RUN / "05_client_sizes.csv", index=False); SIZES.pivot(index="istemci", columns="senaryo", values="benign_cal")

## Eğitim: merkezi, yalnız yerel, FedAvg (Drive'a kaydedilir; devam modunda atlanır)

In [ ]:
d = X["benign_train"].shape[1]
def new_model():
    m = M.build_autoencoder(d, spec["hidden"], spec["dropout"], cfg.learning_rate); return m
def fit(m, Xtr, epochs):
    m.compile(optimizer=tf.keras.optimizers.Adam(cfg.learning_rate), loss="mse")
    m.fit(Xtr, Xtr, epochs=epochs, batch_size=cfg.batch_size, shuffle=True, verbose=0)
def fedavg(parts):
    utils.set_seed(SEED); g = new_model(); local = new_model(); w = g.get_weights()
    n = np.array([len(p) for p in parts], float)
    for r in range(ROUNDS):
        new = []
        for k, p in enumerate(parts):
            local.set_weights(w); fit(local, X["benign_train"][p], LOCAL_EP); new.append(local.get_weights())
        w = [sum(n[k] * new[k][j] for k in range(K)) / n.sum() for j in range(len(w))]
    g.set_weights(w); return g
BUDGET = ROUNDS * LOCAL_EP
MODELS = {}
f = FL_DIR / f"central_seed{SEED}.keras"
if RESUME and f.exists(): MODELS["Merkezi"] = tf.keras.models.load_model(f)
else:
    utils.set_seed(SEED); m = new_model(); fit(m, X["benign_train"], BUDGET); m.save(f); MODELS["Merkezi"] = m
for s in SCEN:
    f = FL_DIR / f"fedavg_{s.replace('=', '')}_seed{SEED}.keras"
    if RESUME and f.exists(): MODELS["FedAvg " + s] = tf.keras.models.load_model(f)
    else: m = fedavg(PARTS[s]["benign_train"]); m.save(f); MODELS["FedAvg " + s] = m
    print(s, "tamam (Drive'a kaydedildi)")
LOCAL = {}
for s in ["IID", "Dirichlet=0.1"]:
    LOCAL[s] = []
    for k, p in enumerate(PARTS[s]["benign_train"]):
        utils.set_seed(SEED + k); m = new_model(); fit(m, X["benign_train"][p], BUDGET); LOCAL[s].append(m)
    print("yalnız yerel", s, "tamam")
n_par = M.count_parameters(MODELS["Merkezi"])
COMM = {"parametre": n_par, "tur": ROUNDS, "istemci": K, "toplam_MB": n_par * 4 * 2 * K * ROUNDS / 1e6}
utils.save_json(COMM, RUN / "05_communication.json"); COMM

## Değerlendirme: global eşik mi, istemci başına eşik mi?
* **Global eşik:** tüm istemcilerin normal-kalibrasyon skorları birleştirilir (sunucu tarafında, yalnızca skorlar).
* **İstemci başına eşik:** her istemci kendi normal-kalibrasyon verisiyle eşik hesaplar.
* Her istemcinin yanlış alarm oranı **kendi** normal-test verisinde ölçülür; görülmemiş saldırı oranları ortak test kümesindedir.

In [ ]:
def scores(m, Xs): return M.ae_score(m, Xs)
rows, SC_SAVE = [], {}
for name, m in list(MODELS.items()) + [("Yalnız yerel " + s, LOCAL[s]) for s in LOCAL]:
    s = name.split(" ", 1)[1] if name.startswith("FedAvg") else ("IID" if name == "Merkezi" else name.replace("Yalnız yerel ", ""))
    parts = PARTS[s]
    for mode in ["global", "istemci başına"]:
        if isinstance(m, list):
            if mode == "global": continue
            mk = m
        else:
            mk = [m] * K
        cal_all = np.concatenate([scores(mk[k], X["benign_cal"][parts["benign_cal"][k]]) for k in range(K)])
        fprs, ha, hu, ka = [], [], [], []
        for k in range(K):
            cal = cal_all if mode == "global" else scores(mk[k], X["benign_cal"][parts["benign_cal"][k]])
            th = C.thresholds(cal, cfg.alpha, cfg.beta)
            bt = scores(mk[k], X["benign_test"][parts["benign_test"][k]])
            fprs.append(np.mean(bt > th["tau_H"]))
            ho = scores(mk[k], X["heldout_test"]); kn = scores(mk[k], X["dev_test"])
            ha.append(np.mean(ho > th["tau_H"])); hu.append(np.mean((ho > th["tau_L"]) & (ho <= th["tau_H"]))); ka.append(np.mean(kn > th["tau_H"]))
        rows.append({"model": name, "eşik": mode, "istemci_FPR_ort": np.mean(fprs), "istemci_FPR_en_kötü": np.max(fprs),
                     "istemci_FPR_en_iyi": np.min(fprs), "en_küçük_n_cal": min(len(p) for p in parts["benign_cal"]),
                     "görülmemiş_ANOMALY": np.mean(ha), "görülmemiş_UNCERTAIN": np.mean(hu), "bilinen_ANOMALY": np.mean(ka)})
    if name == "FedAvg IID":
        SC_SAVE = {sp: scores(m, X[sp]) for sp in ["benign_train", "benign_cal", "benign_test", "dev_test", "heldout_test"]}
np.savez_compressed(FL_DIR / f"fedavg_IID_scores_seed{SEED}.npz", **SC_SAVE)
FL = pd.DataFrame(rows); FL.to_csv(RUN / "05_fl_results.csv", index=False)
FL.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
sub = FL[FL["model"].str.startswith("FedAvg")]
for i, (mode, g) in enumerate(sub.groupby("eşik")):
    xs = np.arange(len(g)) + i * 0.38
    ax.bar(xs, g["istemci_FPR_en_kötü"] * 100, width=0.36, label=f"en kötü istemci · {mode}")
ax.axhline(cfg.alpha * 100, color="k", ls="--", label="α bütçesi")
ax.set_xticks(np.arange(len(sub["model"].unique())) + 0.19); ax.set_xticklabels(sub["model"].unique(), fontsize=8)
ax.set_ylabel("yanlış alarm (%)"); ax.legend(fontsize=8); ax.set_title("Global ve istemci başına eşik: en kötü istemcinin yanlış alarmı")
fig.tight_layout(); fig.savefig(FIG / "05_fl_client_fpr.png", dpi=150)
cfg.save(RUN / "config_used.json"); print("sonuçlar:", RUN)

In [ ]:
# Colab: Drive'a yazılan tüm dosyaların kaydedildiğinden emin ol (oturumu kapatmadan önce mutlaka çalıştırın)
print("Kaydedilen ilerleme:", utils.progress(cfg))
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive kaydedildi ve ayrıldı. Sonraki defter Drive'ı yeniden bağlar.")